# Vaardigheden week 5: gemiddelden, spreiding en de vorm van data

Dit notebook hoort bij het vaardighedenspoor. Elke tabel erin is verzonnen met een vaste seed, zodat de vaardigheden los blijven van het project van deze week en jouw getallen gelijk zijn aan die van iedereen.

Je tekent de vorm van vier processen, berekent twee soorten midden, speelt een spel tot het gemiddelde tot rust komt, ziet hoe een gunstige weddenschap de meeste spelers ruïneert, meet spreiding, betrapt een AI die twee groepen alleen op hun gemiddelde vergeleek, en ziet waarom een kleine verschuiving van een gemiddelde de uitschieters sterk kan veranderen.

**Zo werk je ermee:** voer elke cel op volgorde uit met de afspeelknop of Shift+Enter. Lees de uitvoer voordat je verdergaat. Drie opdrachten "jouw beurt" eindigen elk met een CHECK-cel.

## 1. Een verdeling is de vorm van een proces

Een verdeling legt vast hoe vaak elke waarde voorkomt. Verschillende processen laten verschillende vormen achter. De cel hieronder maakt 10.000 waarden voor elk van vier processen:

- de minuut na het hele uur waarop een telefoontje binnenkomt,
- het gewicht in grammen van een zak van 1 kg die een machine vult,
- het aantal minuten tussen twee bussen,
- een maandinkomen, in een verzonnen munteenheid.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(51)
n = 10_000
shapes = pd.DataFrame({
    "call_minute": rng.uniform(0, 60, n),                     # elke minuut is even waarschijnlijk
    "bag_weight_g": rng.normal(1000, 8, n),                    # veel kleine fouten bij elkaar opgeteld
    "bus_gap_min": rng.exponential(10, n),                     # wachten op iets dat op elk moment kan gebeuren
    "monthly_income": rng.lognormal(np.log(3000), 0.8, n),    # groei die vermenigvuldigt
})
for col in shapes:
    s = shapes[col]
    print(f"{col}: gemiddelde {s.mean():.1f}, mediaan {s.median():.1f}, standaardafwijking {s.std():.1f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, col in zip(axes.flat, shapes.columns):
    ax.hist(shapes[col], bins=60)
    ax.set_title(col)
    ax.set_ylabel("aantal waarden")
plt.tight_layout()
plt.show()

Bekijk de vier histogrammen voordat je verder leest. De belminuten zijn vlak. De zakgewichten vormen een klok. De tijden tussen bussen beginnen hoog en nemen dan af. De inkomens stijgen snel en lopen daarna nog een heel eind uit naar rechts.

**In de staart zitten de verrassingen.** De staart is het uiterste einde van een verdeling. De volgende cel telt hoe vaak elk proces een waarde geeft die meer dan drie standaardafwijkingen boven het gemiddelde ligt. (De standaardafwijking meet de gewone afstand tot het gemiddelde. Deel 6 berekent die met de hand.)

In [ ]:
print("aandeel waarden meer dan 3 standaardafwijkingen boven het gemiddelde:")
for col in shapes:
    s = shapes[col]
    far = (s > s.mean() + 3 * s.std()).mean()
    print(f"  {col}: {far:.2%}")

## 2. Sommen worden klokken

Waarom komt de klok zo vaak voor? Tel veel onafhankelijke kleine effecten bij elkaar op, en het totaal krijgt de vorm van een klok, welke vorm elk effect ook heeft. Eén dobbelsteen is vlak. Kijk wat er gebeurt met het totaal van twee dobbelstenen en van tien dobbelstenen.

In [ ]:
rng = np.random.default_rng(52)
one = rng.integers(1, 7, size=10_000)
two = rng.integers(1, 7, size=(10_000, 2)).sum(axis=1)
ten = rng.integers(1, 7, size=(10_000, 10)).sum(axis=1)
print("meest voorkomende totaal van twee dobbelstenen:", pd.Series(two).mode()[0])

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, values, name in zip(axes, [one, two, ten], ["één dobbelsteen", "totaal van twee dobbelstenen", "totaal van tien dobbelstenen"]):
    ax.hist(values, bins=np.arange(values.min(), values.max() + 2) - 0.5)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 3. Twee middens: het gemiddelde en de mediaan

Het **gemiddelde** telt alle waarden op en deelt door het aantal. De **mediaan** is de middelste waarde als je de waarden sorteert, zodat de helft eronder ligt en de helft erboven.

Een klein kantoor heeft zes medewerkers en één manager. Bereken eerst beide middens met de hand, en laat pandas ze daarna bevestigen.

In [ ]:
pay = pd.Series([2400, 2500, 2600, 2700, 2800, 3000, 14000])   # maandloon: zes medewerkers, dan de manager
print("gemiddeld loon:", round(pay.sum() / len(pay), 2))
print("mediaanloon:", pay.sort_values().iloc[len(pay) // 2])
print("pandas is het ermee eens:", round(pay.mean(), 2), pay.median())

Eén grote waarde trok het gemiddelde ver boven wat zes van de zeven mensen verdienen, en liet de mediaan op zijn plaats. Hetzelfde gebeurt bij de 10.000 verzonnen inkomens uit deel 1.

In [ ]:
income = shapes["monthly_income"]
print(f"gemiddeld inkomen: {income.mean():.0f}")
print(f"mediaaninkomen: {income.median():.0f}")
print(f"aandeel mensen dat minder verdient dan het gemiddelde: {(income < income.mean()).mean():.1%}")

## 4. Het gemiddelde op de lange duur

De **verwachtingswaarde** van een toevallige uitkomst is het gemiddelde dat je zou krijgen als je het heel vaak herhaalt. Vermenigvuldig elke mogelijke uitkomst met de kans erop en tel de resultaten op.

Een spel betaalt drie keer de worp van één dobbelsteen. Bereken de verwachte prijs, speel het spel dan 5.000 keer en kijk hoe het lopende gemiddelde tot rust komt.

In [ ]:
prizes = np.arange(1, 7) * 3            # de zes mogelijke prijzen
print("mogelijke prijzen:", prizes.tolist())
print("verwachte prijs:", (prizes * (1 / 6)).sum())

rng = np.random.default_rng(54)
played = rng.integers(1, 7, size=5000) * 3
running = played.cumsum() / np.arange(1, 5001)
for k in [10, 100, 1000, 5000]:
    print(f"gemiddelde prijs na {k} spellen: {running[k - 1]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(1, 5001), running)
ax.axhline((prizes / 6).sum(), color="grey", linestyle="--")
ax.set_xscale("log")
ax.set_title("Gemiddelde prijs tot nu toe (stippellijn: de verwachte prijs)")
ax.set_xlabel("gespeelde spellen (logaritmische schaal)")
ax.set_ylabel("gemiddelde prijs")
plt.tight_layout()
plt.show()

Een kaartje dat evenveel kost als de verwachte prijs is eerlijk: over veel spellen wint of verliest de speler gemiddeld niets. In het begin dwaalt het lopende gemiddelde. De lange duur is lang.

## 5. Een gunstige weddenschap die de meeste spelers ruïneert

Elke ronde zet een speler alles in op het opgooien van een munt. Kop vermenigvuldigt het geld met 1,5. Munt vermenigvuldigt het met 0,6. De verwachte verandering per ronde is een winst, dus de weddenschap ziet er goed uit. Speel hem met 10.000 spelers gedurende 40 rondes.

In [ ]:
rng = np.random.default_rng(55)
players, rounds, start = 10_000, 40, 100
heads = rng.random((players, rounds)) < 0.5
wealth = start * np.where(heads, 1.5, 0.6).prod(axis=1)     # kop: maal 1,5, munt: maal 0,6
print("spelers:", players, "rondes:", rounds, "startgeld:", start)
print("verwachte verandering per ronde: maal", 0.5 * 1.5 + 0.5 * 0.6)
print(f"gemiddeld eindgeld: {wealth.mean():.1f}")
print(f"mediaan eindgeld: {np.median(wealth):.1f}")
print(f"aandeel spelers dat eindigt met minder dan het begon: {(wealth < start).mean():.1%}")

Het gemiddelde is hoog omdat een paar spelers veel geluk hadden. De mediaanspeler is het meeste geld kwijt. De verwachtingswaarde beschrijft het gemiddelde over veel spelers, en geen enkele speler maakt dat gemiddelde zelf mee.

**Vuistregel:** vertrouw het gemiddelde bij een beslissing die vaak wordt herhaald door iemand die de verliezen onderweg kan opvangen. Wantrouw het wanneer één slechte reeks het spel kan beëindigen.

## 6. Spreiding: hetzelfde gemiddelde, andere werelden

De **variantie** is de gemiddelde gekwadrateerde afstand tot het gemiddelde. De **standaardafwijking** is de wortel daaruit, dus die heeft dezelfde eenheid als de data en lees je als de gewone afstand tot het gemiddelde.

Bereken beide met de hand voor vijf wachttijden. Er is één valkuil: software deelt de som van de gekwadrateerde afstanden door het aantal waarden (n) of door één minder (n - 1). pandas gebruikt n - 1, en numpy gebruikt n.

In [ ]:
w = pd.Series([20, 25, 30, 35, 40])            # vijf wachttijden, in minuten
dist = w - w.mean()
print("afstanden tot het gemiddelde:", dist.tolist())
print("variantie, gedeeld door n:", (dist ** 2).sum() / len(w))
print("variantie, gedeeld door n - 1:", (dist ** 2).sum() / (len(w) - 1))
print("pandas w.var():", w.var(), "  numpy np.var(w):", np.var(w.to_numpy()))
print("standaardafwijking, gedeeld door n:", round(np.sqrt((dist ** 2).sum() / len(w)), 2))

Met vijf waarden geven de twee manieren van delen duidelijk verschillende antwoorden. Met duizenden waarden zijn ze het tot op twee decimalen eens. Welke je ook gebruikt, zeg welke het is.

**Percentielen** beschrijven spreiding zonder een vorm aan te nemen. Het 90e percentiel is de waarde waar 90% van de waarden onder ligt. De afstand van het 10e tot het 90e percentiel omvat de middelste 80% van de waarden.

Maak nu kennis met twee klinieken. Elke kliniek noteert de wachttijd van 2.000 patiënten.

In [ ]:
rng = np.random.default_rng(56)
waits = pd.DataFrame({
    "clinic": ["A"] * 2000 + ["B"] * 2000,
    "wait_min": np.concatenate([rng.normal(30, 5, 2000),           # kliniek A: iedereen wacht ongeveer even lang
                                rng.exponential(30, 2000)]).round(1),   # kliniek B: de meesten snel, een paar heel lang
})
print(waits.groupby("clinic").size())
print(waits.sample(6, random_state=1))

## 7. Lees code die een AI-assistent schreef

We vroegen een AI-assistent: *"Wachten patiënten langer bij kliniek A of bij kliniek B?"* De assistent schreef de cel hieronder. Voer hem uit en beslis dan of de conclusie klopt.

In [ ]:
# Geschreven door een AI-assistent
avg = waits.groupby("clinic")["wait_min"].mean()
print(f"Kliniek A: gemiddeld {avg['A']:.1f} minuten")
print(f"Kliniek B: gemiddeld {avg['B']:.1f} minuten")
diff = abs(avg["A"] - avg["B"])
print(f"Het verschil is {diff:.1f} minuten, dus patiënten wachten bij beide klinieken ongeveer even lang.")

De berekening klopt, en toch is de conclusie fout. De assistent bracht elke kliniek terug tot één getal en keek nooit naar de vorm. Een patiënt ervaart niet de gemiddelde wachttijd. Een patiënt ervaart één wachttijd, en de vraag is hoe lang die kan duren.

De volgende twee cellen laten zien wat de gemiddelden verbergen.

In [ ]:
for clinic, s in waits.groupby("clinic")["wait_min"]:
    print(f"kliniek {clinic}: mediaan {s.median():.1f}, standaardafwijking {s.std():.1f}, "
          f"langer dan 60 minuten {(s > 60).mean():.1%}, langste {s.max():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.arange(0, waits["wait_min"].max() + 5, 5)
for clinic, s in waits.groupby("clinic")["wait_min"]:
    ax.hist(s, bins=bins, alpha=0.6, label=f"kliniek {clinic}")
ax.axvline(60, color="grey", linestyle="--")
ax.set_title("Wachttijden bij twee klinieken met hetzelfde gemiddelde (stippellijn: één uur)")
ax.set_xlabel("wachttijd in minuten")
ax.set_ylabel("patiënten")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Jouw beurt: een samenvatting die de vorm niet verbergt

Vraag je AI-assistent om code die een tabel met de naam `summary` maakt, met één rij per kliniek (de index is `"A"` en `"B"`) en vijf kolommen: `mean`, `median`, `sd`, `p90` (het 90e percentiel) en `share_over_60` (het aandeel patiënten dat langer dan 60 minuten wachtte, als fractie tussen 0 en 1).

Plak de code in de cel hieronder, voer hem uit en voer daarna de CHECK-cel uit. Vergelijk één rij met de uitvoer van deel 7 voordat je de tabel vertrouwt.

In [ ]:
# jouw beurt 1: maak een tabel met de naam summary, met één rij per kliniek en de kolommen mean, median, sd, p90 en share_over_60
summary = None

In [ ]:
# CHECK 1
assert summary is not None, "summary is nog leeg"
for clinic, s in waits.groupby("clinic")["wait_min"]:
    row = summary.loc[clinic]
    assert abs(row["mean"] - s.mean()) < 0.01, f"mean voor kliniek {clinic}"
    assert abs(row["median"] - s.median()) < 0.01, f"median voor kliniek {clinic}"
    assert abs(row["sd"] - s.std()) < 0.05, f"sd voor kliniek {clinic}"         # n of n - 1 slagen allebei met 2.000 waarden
    assert abs(row["p90"] - s.quantile(0.9)) < 0.5, f"p90 voor kliniek {clinic}"
    assert abs(row["share_over_60"] - (s > 60).mean()) < 0.001, f"share_over_60 voor kliniek {clinic} (een fractie, geen percentage)"
print("CHECK 1 geslaagd")
print(summary.round(3))

## 9. Het gemiddelde van veel is stabieler dan één waarde

Losse wachttijden bij kliniek B lopen sterk uiteen. En het gemiddelde van 16 wachttijden, of van 256? De cel hieronder trekt 5.000 gemiddelden voor elke grootte en meet hoe sterk ze uiteenlopen.

In [ ]:
rng = np.random.default_rng(57)
print(f"standaardafwijking van losse wachttijden: {rng.exponential(30, 1_000_000).std():.1f}")
for k in [1, 4, 16, 64, 256]:
    averages = rng.exponential(30, size=(5000, k)).mean(axis=1)    # 5.000 gemiddelden, elk van k wachttijden
    print(f"n = {k}: standaardafwijking van de gemiddelden {averages.std():.2f}, wortelregel 30 / sqrt({k}) = {30 / np.sqrt(k):.2f}")

In [ ]:
rng = np.random.default_rng(58)
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharex=True)
for ax, k in zip(axes, [1, 16, 256]):
    ax.hist(rng.exponential(30, size=(5000, k)).mean(axis=1), bins=60)
    ax.set_title("losse wachttijden" if k == 1 else f"gemiddelden van {k} wachttijden")
    ax.set_xlabel("minuten")
plt.tight_layout()
plt.show()

**De wortelregel:** de spreiding van een gemiddelde is gelijk aan de spreiding van losse waarden gedeeld door de wortel uit het aantal waarden in dat gemiddelde. Vier keer zoveel data halveert de spreiding van het gemiddelde. De gemiddelden worden ook een klok, om de reden uit deel 2: een gemiddelde is een som gedeeld door een aantal.

Twee kanttekeningen. De regel gaat over het gemiddelde, niet over losse waarden: de patiënten van kliniek B wachten nog even onvoorspelbaar als eerst. En de regel neemt aan dat de waarden onafhankelijk zijn. Bij waarden die samen bewegen, zoals de verkoop van één winkel op opeenvolgende dagen, krimpt de spreiding van het gemiddelde langzamer.

## 10. Jouw beurt: toets de regel op kliniek A

De wachttijden van kliniek A volgen een klok met een gemiddelde van 30 minuten en een standaardafwijking van 5 minuten. Vraag je AI-assistent om code die een pandas-Series met de naam `spread_of_avg` maakt, met als index n = 1, 4, 16 en 64, met daarin de standaardafwijking van 5.000 gemiddelden van n wachttijden, getrokken met `rng.normal(30, 5, ...)`.

Schrijf op wat de wortelregel voor elke n voorspelt voordat je de CHECK uitvoert.

In [ ]:
# jouw beurt 2: maak een Series met de naam spread_of_avg, met als index n = 1, 4, 16, 64
spread_of_avg = None

In [ ]:
# CHECK 2
assert spread_of_avg is not None, "spread_of_avg is nog leeg"
for k in [1, 4, 16, 64]:
    rule = 5 / np.sqrt(k)
    assert abs(spread_of_avg[k] - rule) / rule < 0.05, f"n = {k} ligt meer dan 5% van de wortelregel af"
print("CHECK 2 geslaagd:", spread_of_avg.round(2).to_dict())

## 11. Een kleine verschuiving van het gemiddelde, een grote verandering in de staart

Het dagelijkse elektriciteitsverbruik van een gebouw volgt een klok. Noem een dag "hoog" als die boven het oude 90e percentiel ligt, zodat één op de tien dagen hoog is. Nu verbruikt elke dag iets meer. Kijk naar het gemiddelde en naar de hoge dagen.

In [ ]:
rng = np.random.default_rng(59)
before = rng.normal(50, 10, 100_000)          # dagverbruik in kWh
threshold = np.percentile(before, 90)          # een hoge dag ligt boven het oude 90e percentiel
after = before + 3                             # elke dag verbruikt 3 kWh meer
wider = 53 + 1.2 * (before - 50)               # 3 kWh meer, en de spreiding 20% breder
print(f"drempel voor een hoge dag: {threshold:.1f} kWh")
print(f"het gemiddelde steeg met {(after.mean() - before.mean()) / before.mean():.1%}")
print(f"hoge dagen ervoor: {(before > threshold).mean():.1%}")
print(f"hoge dagen na de verschuiving: {(after > threshold).mean():.1%}")
print(f"hoge dagen na de verschuiving met een bredere spreiding: {(wider > threshold).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.arange(0, 101, 1)
ax.hist(before, bins=bins, alpha=0.5, density=True, label="ervoor")
ax.hist(wider, bins=bins, alpha=0.5, density=True, label="erna, met een bredere spreiding")
ax.axvline(threshold, color="grey", linestyle="--")
ax.set_title("Dagelijks elektriciteitsverbruik (stippellijn: de drempel voor een hoge dag)")
ax.set_xlabel("kWh per dag")
ax.set_ylabel("aandeel dagen per kWh")
ax.legend()
plt.tight_layout()
plt.show()

Veel dagen lagen net onder de drempel. Een klein duwtje brengt ze erover, dus het aandeel boven de drempel groeit relatief veel sneller dan het gemiddelde. Als een vraag over uitschieters gaat, meet dan de uitschieters zelf.

## 12. Jouw beurt: hoe snel groeit de staart?

Vraag je AI-assistent om code die een dictionary met de naam `shares` maakt. Die koppelt elke verschuiving van 1, 2 en 5 kWh aan het aandeel dagen boven `threshold` nadat elke waarde in `before` met die verschuiving is verhoogd. Voorspel de volgorde van de drie aandelen voordat je de CHECK uitvoert.

In [ ]:
# jouw beurt 3: maak een dict met de naam shares die elke verschuiving (1, 2 en 5) koppelt aan het aandeel dagen boven threshold
shares = None

In [ ]:
# CHECK 3
assert shares is not None, "shares is nog leeg"
for shift in [1, 2, 5]:
    assert abs(shares[shift] - (before + shift > threshold).mean()) < 1e-9, f"het aandeel bij een verschuiving van {shift}"
assert shares[1] < shares[2] < shares[5]
print("CHECK 3 geslaagd:", {k: round(float(v), 3) for k, v in shares.items()})
print("een verschuiving van 5 kWh maakt hoge dagen", round(shares[5] / (before > threshold).mean(), 2), "keer zo vaak voorkomend")

## 13. Wat je meeneemt

Doorloop vier stappen voordat je twee reeksen getallen vergelijkt.

1. Teken beide histogrammen met dezelfde klassen.
2. Kies een midden dat bij de vorm past: de mediaan bij een lange staart, het gemiddelde bij een klok.
3. Geef naast het midden een spreiding: een standaardafwijking, of het 10e en 90e percentiel.
4. Gaat de vraag over uitschieters, tel de uitschieters dan zelf, met een drempel die je kiest voordat je naar het resultaat kijkt.